# Bambara SLM: Stage 1 CPT on Kaggle (resume-capable)

**Setup (once)**
1. Kaggle account with a **verified phone number** (needed for GPU and internet).
2. Add two **Datasets** (Kaggle → Datasets → New dataset, upload the zip):
   - `bambara-slm-code` ← `dist/bambara_slm_colab.zip` (code + CPT data; same bundle as for Colab)
   - `bambara-slm-run-with-nllb` ← the run folder from Google Drive `MyDrive/bambara-SLM/runs/cpt_with_nllb`
     (download it from Drive as a zip). This holds the checkpoint training resumes from.
3. In this notebook: **+ Add Input** (right panel) → *Your Work* → attach **both** datasets (the panel must not say "No input"). **Settings**: Accelerator = **GPU T4** (one GPU is used), Internet = **On**.
4. Run it in the background: **Save Version → Save & Run All (Commit)**. You can close the browser.
   It stops cleanly after `MAX_HOURS`, below Kaggle's 12 h limit, and saves a checkpoint.

**After the run:** open the version → **Output** and download `resume_pack_<variant>.zip`
(everything needed to continue, on Kaggle or Colab) and, if training finished, `results_<variant>.zip`.

## Setup (one cell): inputs, code, packages, restore checkpoint
Run it once interactively to check it ends with `checkpoints: [...]`, then **Save Version → Save & Run All (Commit)**.


In [ ]:
import glob, os, shutil, subprocess, time, zipfile
VARIANT   = "with_nllb"               # "no_nllb" for the comparison run
WORK      = "/kaggle/working"
RUN_DIR   = f"{WORK}/runs/cpt_{VARIANT}"
MAX_HOURS = 11.0                         # stop cleanly before Kaggle's 12 h limit
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # use ONE of the two T4s (same effective batch as before)

def find_input(keyword, required=True):
    hits = [p for p in glob.glob("/kaggle/input/**/", recursive=True) if keyword in p.lower()]
    if not hits:
        assert not required, f"no input matching '{keyword}': click 'Add Input' and attach the dataset"
        return None
    return sorted(hits, key=len)[0].rstrip("/")
CODE_IN = find_input("bambara-slm-code")
# previous run of THIS variant (to resume): dataset named bambara-slm-run-with-nllb / bambara-slm-run-no-nllb
RUN_IN  = find_input("bambara-slm-run-" + VARIANT.replace("_", "-"), required=False)

# 1. code -> /tmp/code
shutil.rmtree("/tmp/code", ignore_errors=True)
zips = glob.glob(f"{CODE_IN}/**/*.zip", recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall("/tmp/code")
else:
    shutil.copytree(CODE_IN, "/tmp/code")
CODE = os.path.dirname(sorted(glob.glob("/tmp/code/**/pyproject.toml", recursive=True), key=len)[0])
assert "class TimeBudget" in open(f"{CODE}/src/bslm/train/cpt.py").read(), "old bundle: upload the new bambara_slm_colab.zip"
print("code:", CODE, os.listdir(f"{CODE}/data/cpt"))

# 2. packages (conflict warnings about bigframes/cudf are harmless)
subprocess.run(f'cd "{CODE}" && pip install -q -r requirements-gpu.txt', shell=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)   # breaks peft; unused here

# 3. restore the previous run of this variant (checkpoint + history), if attached; otherwise start fresh
if RUN_IN:
    t = time.time()
    os.makedirs(f"{WORK}/runs", exist_ok=True)
    zips = glob.glob(f"{RUN_IN}/**/*.zip", recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall(f"{WORK}/runs")
        src = os.path.dirname(sorted(glob.glob(f"{WORK}/runs/**/start_eval.json", recursive=True), key=len)[0])
        if os.path.abspath(src) != os.path.abspath(RUN_DIR):
            shutil.move(src, RUN_DIR)
    else:
        src = os.path.dirname(sorted(glob.glob(f"{RUN_IN}/**/start_eval.json", recursive=True), key=len)[0])
        shutil.copytree(src, RUN_DIR, dirs_exist_ok=True)
    print(f"restored in {time.time()-t:.0f}s:", sorted(os.listdir(RUN_DIR)))
    print("checkpoints:", sorted(os.listdir(f"{RUN_DIR}/checkpoints")))
else:
    print(f"no previous run attached for {VARIANT}: FRESH START from the base model")

# 4. optional Hugging Face token (Kaggle secret named HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

## Train (resumes automatically from the restored checkpoint)

In [ ]:
os.makedirs(RUN_DIR, exist_ok=True)   # fresh runs: the folder must exist before tee opens train.log
os.environ.update({"PYTHONPATH": f"{CODE}/src", "BSLM_RUN_DIR": RUN_DIR, "BSLM_MAX_HOURS": str(MAX_HOURS),
                   "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"})
rc = subprocess.call(f'cd "{CODE}" && python -m bslm.train.cpt --config configs/train/cpt_{VARIANT}.yaml 2>&1 | tee -a "{RUN_DIR}/train.log"; exit ${{PIPESTATUS[0]}}',
                     shell=True, executable="/bin/bash")
print("exit code:", rc, {0: "FINISHED: merged model saved", 2: "finished, but no checkpoint met the +10% limit: tell Claude",
                         3: "time budget reached: download resume_pack and continue in a new session"}.get(rc, "ERROR: see log above"))

## Pack outputs


In [ ]:
# pack outputs: resume_pack (everything needed to continue) and, if finished, results (merged model + reports)
import json
os.makedirs(RUN_DIR, exist_ok=True)
os.chdir(WORK)
!cd "{WORK}/runs" && zip -qr "{WORK}/resume_pack_{VARIANT}.zip" "cpt_{VARIANT}" -x "cpt_{VARIANT}/merged/*"
if os.path.exists(f"{RUN_DIR}/merged"):
    !cd "{WORK}/runs" && zip -qr "{WORK}/results_{VARIANT}.zip" "cpt_{VARIANT}" -x "cpt_{VARIANT}/checkpoints/*"
    print(json.dumps(json.load(open(f"{RUN_DIR}/selection.json")), indent=2))
!ls -lh {WORK}/*.zip
print(open(f"{RUN_DIR}/eval_history.json").read()[-1500:] if os.path.exists(f"{RUN_DIR}/eval_history.json") else "no eval history yet")